# Malinois MPRA 4: summed motifbreakR, with every call weighted by its P(bound)

The diagnosis notebook found that summing alleleDiff over all of a variant's motifbreakR calls doubles its Pearson
with Δ (HepG2 0.069 → 0.146), because variants that break several motifs have larger effects. Every earlier CAV rule
kept one call per variant. Here each call is weighted by the CAV's P(bound) of its occurrence and then summed, and
compared with the plain sum on the same variants and calls:

| rule | prediction per variant | uses MPRA labels? |
|---|---|---|
| summed motifbreakR (baseline) | Σ alleleDiff | no |
| **Σ alleleDiff × P** (main rule, fixed in advance) | each motif change counts as much as its site is likely bound | no |
| Σ alleleDiff × P percentile | P as its percentile among the calls of the same TF (removes between-TF differences in P) | no |
| Σ alleleDiff × (0.5 + P percentile) | the same, but a call with low P still counts half | no |
| learned mix | Δ ~ Σ alleleDiff + Σ alleleDiff × (P percentile − 0.5), fitted on one chromosome half, scored on the other | 2 weights |

Scores per cell (HepG2, K562, SK-N-SH) and variant set (all / active): Pearson, Spearman, AUROC for clear effects
(|Δ| ≥ 2 SE), each as gain over the summed baseline. Null N2: P shuffled among the calls of the same TF (200 times,
the whole computation redone). Bootstrap 95% CI of the main rule's gain, and of its HepG2/K562 − SK-N-SH difference.
TF tiers by held-out ChIP AUC (all, ≥ 0.70, ≥ 0.75, ≥ 0.80) as in `Malinois MPRA 2`. CPU only, about 15–30 minutes.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Load

In [2]:
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, time
import numpy as np, pandas as pd
from scipy import stats
from sklearn.metrics import roc_auc_score

MAL = f"{WORK_DIR}/malinois"
POOL = f"{WORK_DIR}/fcav_pool"
OUT = f"{MAL}/eval_summed"
os.makedirs(OUT, exist_ok=True)
CELLS3 = ["HepG2", "K562", "SKNSH"]
TIERS = [("all TFs", 0.0), ("AUC >= 0.70", 0.70), ("AUC >= 0.75", 0.75), ("AUC >= 0.80", 0.80)]
ACTIVE_MIN, Z_CLEAR = 1.0, 2.0
N_PERM, N_BOOT, LAYER = 200, 300, 24
MAIN = "Σ AD × P"
pd.set_option("display.width", 250)
fmt = lambda x: f"{x:+.3f}"

V = pd.read_csv(f"{MAL}/variants.csv.gz", index_col="vi", dtype={"chrom": str}, low_memory=False)
H = pd.read_csv(f"{MAL}/hits.csv")
CALLS = pd.read_csv(f"{MAL}/calls.csv")
PD_ = f"{MAL}/p_nt500m"
PU = pd.concat([pd.read_csv(f"{PD_}/{f}") for f in os.listdir(PD_) if f.endswith(".csv") and not f.endswith(".part.csv")])
H = H[~H["leak"]].merge(PU, on="uid")
PVM = H.groupby(["vi", "motif"])["P"].max().rename("P").reset_index()
LALL = CALLS.merge(PVM, on=["vi", "motif"])
AUC = {}
for m in LALL["motif"].unique():
    z = np.load(f"{POOL}/fits/nt500m/{m}.npz", allow_pickle=False)
    AUC[m] = float(np.asarray(z["auc"])[[int(l) for l in z["layers"]].index(LAYER)])
LALL["auc"] = LALL["motif"].map(AUC)
print(f"{len(LALL):,} calls with a P(bound) on {LALL['vi'].nunique():,} variants")

115,254 calls with a P(bound) on 67,447 variants


## 2. Rules, scores, nulls

In [3]:
def pear(a, b):
    a, b = a - a.mean(), b - b.mean()
    return float((a * b).sum() / np.sqrt((a * a).sum() * (b * b).sum()))


def tf_pct(p, g):
    o = np.lexsort((p, g))
    gs = g[o]
    st = np.r_[0, np.flatnonzero(gs[1:] != gs[:-1]) + 1]
    cnt = np.diff(np.r_[st, len(gs)])
    q = np.empty(len(p)); q[o] = (np.arange(len(gs)) - np.repeat(st, cnt) + 0.5) / np.repeat(cnt, cnt)
    return q


def context(L):
    L = L.sort_values(["vi", "motif"]).reset_index(drop=True)
    VI = np.unique(L["vi"].values)
    X = dict(VI=VI, cv=np.searchsorted(VI, L["vi"].values), cm=pd.factorize(L["motif"])[0],
             AD=L["alleleDiff"].values, P0=L["P"].values)
    X["base"] = np.argsort(X["cm"], kind="stable")
    chrom = V.loc[VI, "chrom"].astype(str).values
    X["FOLD"] = np.array([int(ch) % 2 if ch.isdigit() else 0 for ch in chrom])
    X["Y"] = {c: V.loc[VI, f"d_{c}"].values for c in CELLS3}
    X["CL"] = {c: (np.abs(X["Y"][c]) / V.loc[VI, f"se_{c}"].values >= Z_CLEAR) for c in CELLS3}
    X["SETS"] = {(c, s): np.isfinite(X["Y"][c]) & ((V.loc[VI, f"act_{c}"].values >= ACTIVE_MIN) if s == "active" else True)
                 for c in CELLS3 for s in ("all", "active")}
    return X


def shuffle_tf(X, rng):
    o = np.lexsort((rng.random(len(X["P0"])), X["cm"]))
    out = np.empty_like(X["P0"]); out[X["base"]] = X["P0"][o]
    return out


def rule_scores(X, p):
    # per variant: the baseline and the four rules (the learned mix is returned as its two columns)
    n, cv, ad = len(X["VI"]), X["cv"], X["AD"]
    q = tf_pct(p, X["cm"])
    s = lambda w: np.bincount(cv, ad * w, minlength=n)
    return {"summed motifbreakR": s(1.0), "Σ AD × P": s(p), "Σ AD × P percentile": s(q),
            "Σ AD × (0.5 + P percentile)": s(0.5 + q), "_mix": s(q - 0.5)}


def learned(base, mix, y, f):
    Z = np.column_stack([np.ones(len(y)), base, mix])
    out = np.zeros(len(y))
    for k in (0, 1):
        tr, te = f != k, f == k
        out[te] = Z[te] @ np.linalg.lstsq(Z[tr], y[tr], rcond=None)[0]
    return out


RULES = ["Σ AD × P", "Σ AD × P percentile", "Σ AD × (0.5 + P percentile)", "learned mix"]


def evaluate(X, p, full=True):
    # {(cell, set): {rule: (Pearson gain, AUROC gain)}} plus, if full, the levels and Spearman
    S = rule_scores(X, p)
    out = {}
    for (c, s), m in X["SETS"].items():
        y, cl, b = X["Y"][c][m], X["CL"][c][m], S["summed motifbreakR"][m]
        preds = {r: S[r][m] for r in RULES[:3]}
        preds["learned mix"] = learned(b, S["_mix"][m], y, X["FOLD"][m])
        pb, ab = pear(b, y), roc_auc_score(cl, np.abs(b))
        res = {}
        for r, pr in preds.items():
            res[r] = (pear(pr, y) - pb, roc_auc_score(cl, np.abs(pr)) - ab)
        if full:
            res["_levels"] = {"n": int(m.sum()), "Pearson summed": pb, "AUROC summed": ab,
                              "Spearman summed": stats.spearmanr(b, y)[0],
                              **{f"Spearman gain {r}": stats.spearmanr(pr, y)[0] - stats.spearmanr(b, y)[0] for r, pr in preds.items()}}
        out[c, s] = res
    return out


def main_gain(X, idx, c, s):
    m = idx[X["SETS"][c, s][idx]]
    S = X["_S"]
    y = X["Y"][c][m]
    return pear(S[MAIN][m], y) - pear(S["summed motifbreakR"][m], y)

## 3. Every tier

In [4]:
ROWS, CROWS = [], []
t0 = time.time()
for name, thr in TIERS:
    X = context(LALL[LALL["auc"] >= thr])
    X["_S"] = rule_scores(X, X["P0"])
    rng = np.random.default_rng(7)
    obs = evaluate(X, X["P0"])
    null = [evaluate(X, shuffle_tf(X, rng), full=False) for _ in range(N_PERM)]
    for key, res in obs.items():
        c, s = key
        row = {"threshold": name, "cell": c, "variants": s, **res["_levels"]}
        for r in RULES:
            nl = np.array([n_[key][r] for n_ in null])
            row[f"Pearson gain {r}"], row[f"p N2 Pearson {r}"] = res[r][0], np.mean(nl[:, 0] >= res[r][0])
            row[f"AUROC gain {r}"], row[f"p N2 AUROC {r}"] = res[r][1], np.mean(nl[:, 1] >= res[r][1])
        m = np.flatnonzero(X["SETS"][key])
        bs = [main_gain(X, rng.choice(m, len(m)), c, s) for _ in range(N_BOOT)]
        row["main Pearson gain CI low"], row["main Pearson gain CI high"] = np.percentile(bs, [2.5, 97.5])
        ROWS.append(row)
    for c in ("HepG2", "K562"):
        for s in ("all", "active"):
            u = np.flatnonzero(X["SETS"][c, s] | X["SETS"]["SKNSH", s])
            allv = np.arange(len(X["VI"]))
            d = [main_gain(X, j, c, s) - main_gain(X, j, "SKNSH", s) for j in (rng.choice(u, len(u)) for _ in range(N_BOOT))]
            CROWS.append({"threshold": name, "contrast": f"{c} - SK-N-SH", "variants": s,
                          "difference in main Pearson gain": main_gain(X, allv, c, s) - main_gain(X, allv, "SKNSH", s),
                          "95% CI low": np.percentile(d, 2.5), "95% CI high": np.percentile(d, 97.5)})
    print(f"{name}: {len(np.unique(X['cm']))} TFs, {len(X['VI']):,} variants  [{(time.time() - t0) / 60:.1f} min]", flush=True)
R = pd.DataFrame(ROWS).set_index(["threshold", "cell", "variants"])
CR = pd.DataFrame(CROWS).set_index(["threshold", "contrast", "variants"])
R.to_csv(f"{OUT}/summed_rules.csv"); CR.to_csv(f"{OUT}/summed_contrast.csv")

print(f"\np N2 = share of {N_PERM} TF-wise shuffles of P with at least this gain\n")
print(f"=== main rule ({MAIN}) vs summed motifbreakR")
print(R[["n", "Pearson summed", f"Pearson gain {MAIN}", f"p N2 Pearson {MAIN}", "main Pearson gain CI low", "main Pearson gain CI high",
         "AUROC summed", f"AUROC gain {MAIN}", f"p N2 AUROC {MAIN}", "Spearman summed", f"Spearman gain {MAIN}"]].to_string(float_format=fmt))
print("\n=== the other rules: Pearson gain (p N2) and AUROC gain (p N2)")
print(R[[f"{k} {r}" for r in RULES[1:] for k in ("Pearson gain", "p N2 Pearson", "AUROC gain", "p N2 AUROC")]].to_string(float_format=fmt))
print(f"\n=== cell contrast of the main rule's Pearson gain (bootstrap 95% CI)")
print(CR.to_string(float_format=fmt))

all TFs: 251 TFs, 67,447 variants  [2.4 min]
AUC >= 0.70: 143 TFs, 48,086 variants  [4.1 min]
AUC >= 0.75: 95 TFs, 35,708 variants  [5.4 min]
AUC >= 0.80: 50 TFs, 22,483 variants  [6.3 min]

p N2 = share of 200 TF-wise shuffles of P with at least this gain

=== main rule (Σ AD × P) vs summed motifbreakR
                                n  Pearson summed  Pearson gain Σ AD × P  p N2 Pearson Σ AD × P  main Pearson gain CI low  main Pearson gain CI high  AUROC summed  AUROC gain Σ AD × P  p N2 AUROC Σ AD × P  Spearman summed  Spearman gain Σ AD × P
threshold   cell  variants                                                                                                                                                                                                                           
all TFs     HepG2 all       67447          +0.144                 -0.006                 +0.000                    -0.010                     -0.003        +0.552               +0.015               +0.00

## End

Outputs in `malinois/eval_summed/`: `summed_rules.csv`, `summed_contrast.csv`.